In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
tmdb_df = pd.read_csv("../data/tmdb_movies.csv")

feature_data = np.load("../data/tmdb_movie_features.npz")

movie_ids = feature_data["movie_ids"]
movie_features = feature_data["features"]

print("Movie IDs:", movie_ids.shape)
print("Movie features:", movie_features.shape)

Movie IDs: (86250,)
Movie features: (86250, 561)


In [3]:
ratings = pd.read_csv("..\\data\\ml-32m\\ratings.csv")

print("Ratings shape:", ratings.shape)
ratings.head()

KeyboardInterrupt: 

In [ ]:
tmdb_movie_ids = set(movie_ids)

matched = ratings["movieId"].isin(tmdb_movie_ids)

print("Total ratings:", len(ratings))
print("Matched ratings:", matched.sum())
print("Rating coverage:", f"{matched.mean():.2%}")

print()
print("Unique MovieLens movies:", ratings["movieId"].nunique())
print(
    "Movies with TMDB features:",
    ratings.loc[matched, "movieId"].nunique()
)

Total ratings: 32000204
Matched ratings: 31918409
Rating coverage: 99.74%

Unique MovieLens movies: 84432
Movies with TMDB features: 83128


### Create Movie-Level Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Movies that have both MovieLens ratings and TMDB features
matched_movie_ids = ratings.loc[
    matched, "movieId"
].unique()

# 60% train, 40% temporary
train_movies, temp_movies = train_test_split(
    matched_movie_ids,
    test_size=0.4,
    random_state=42
)

# Split remaining 40% equally → 20% validation, 20% test
val_movies, test_movies = train_test_split(
    temp_movies,
    test_size=0.5,
    random_state=42
)

print("Train movies:", len(train_movies))
print("Validation movies:", len(val_movies))
print("Test movies:", len(test_movies))

Train movies: 49876
Validation movies: 16626
Test movies: 16626


# Small-Sample Predictive Modeling

This section evaluates whether the engineered TMDB movie features contain useful predictive information for MovieLens ratings.

A small sample of MovieLens ratings is used for initial experimentation. Multiple regression models will be trained and compared using the same train, validation, and test splits.

Models:
- Linear Regression
- Random Forest
- Neural Network

In [ ]:
ratings_sample = ratings.loc[matched].sample(
    n=100000,
    random_state=42
).copy()


### Train / Validation / Test Split

In [ ]:
from sklearn.model_selection import train_test_split

sample_movie_ids = ratings_sample["movieId"].unique()

# 60% train, 40% temporary
train_movies, temp_movies = train_test_split(
    sample_movie_ids,
    test_size=0.4,
    random_state=42
)

# 20% validation, 20% test
val_movies, test_movies = train_test_split(
    temp_movies,
    test_size=0.5,
    random_state=42
)

train_ratings = ratings_sample[
    ratings_sample["movieId"].isin(train_movies)
].copy()

val_ratings = ratings_sample[
    ratings_sample["movieId"].isin(val_movies)
].copy()

test_ratings = ratings_sample[
    ratings_sample["movieId"].isin(test_movies)
].copy()

print("Train ratings:", len(train_ratings))
print("Validation ratings:", len(val_ratings))
print("Test ratings:", len(test_ratings))

print()
print("Train movies:", len(train_movies))
print("Validation movies:", len(val_movies))
print("Test movies:", len(test_movies))

Train ratings: 59459
Validation ratings: 19164
Test ratings: 21377

Train movies: 6657
Validation movies: 2219
Test movies: 2220


### Model Input Features

In [ ]:
# Map movieId to its row in the TMDB feature matrix
movie_id_to_index = {
    movie_id: idx
    for idx, movie_id in enumerate(movie_ids)
}

def create_xy(rating_df):
    indices = rating_df["movieId"].map(movie_id_to_index).to_numpy()

    X = movie_features[indices]
    y = rating_df["rating"].to_numpy()

    return X, y


X_train, y_train = create_xy(train_ratings)
X_val, y_val = create_xy(val_ratings)
X_test, y_test = create_xy(test_ratings)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

Train: (59459, 784) (59459,)
Validation: (19164, 784) (19164,)
Test: (21377, 784) (21377,)


## Linear Regression

A Linear Regression model is used as the first baseline to evaluate how well the engineered TMDB movie features can predict MovieLens ratings.

The model is trained on the training set and evaluated on the validation set using RMSE, MAE, and R².

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_val_pred = lr_model.predict(X_val)
y_test_pred = lr_model.predict(X_test)

# Evaluation function
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    return rmse, mae, r2

# Evaluate 
# EVALUATE TRAINING TOO !!
val_rmse, val_mae, val_r2 = evaluate_model(y_val, y_val_pred)
test_rmse, test_mae, test_r2 = evaluate_model(y_test, y_test_pred)

print("Linear Regression Results")
print("-------------------------")

print(f"Validation RMSE: {val_rmse:.4f}")
print(f"Validation MAE:  {val_mae:.4f}")
print(f"Validation R²:   {val_r2:.4f}")

print()

print(f"Test RMSE:       {test_rmse:.4f}")
print(f"Test MAE:        {test_mae:.4f}")
print(f"Test R²:         {test_r2:.4f}")

Linear Regression Results
-------------------------
Validation RMSE: 1.0525
Validation MAE:  0.8321
Validation R²:   0.0213

Test RMSE:       1.0458
Test MAE:        0.8218
Test R²:         0.0202


## Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Train Random Forest
rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Predictions
y_val_pred_rf = rf_model.predict(X_val)
y_test_pred_rf = rf_model.predict(X_test)

# Evaluation
val_rmse_rf, val_mae_rf, val_r2_rf = evaluate_model(
    y_val, y_val_pred_rf
)

test_rmse_rf, test_mae_rf, test_r2_rf = evaluate_model(
    y_test, y_test_pred_rf
)

print("Random Forest Results")
print("---------------------")

print(f"Validation RMSE: {val_rmse_rf:.4f}")
print(f"Validation MAE:  {val_mae_rf:.4f}")
print(f"Validation R²:   {val_r2_rf:.4f}")

print()

print(f"Test RMSE:       {test_rmse_rf:.4f}")
print(f"Test MAE:        {test_mae_rf:.4f}")
print(f"Test R²:         {test_r2_rf:.4f}")

Random Forest Results
---------------------
Validation RMSE: 1.0522
Validation MAE:  0.8486
Validation R²:   0.0218

Test RMSE:       1.0447
Test MAE:        0.8425
Test R²:         0.0222
